# Week 11 - 2026-06-17 실습

## 📅 오늘 학습 주제: Theme 44: Ensemble Retriever 및 하이브리드 검색(Hybrid Search) 실습
- **학습 목표**:
  - 어제 실습에서 확인한 단순 Vector Store 기반 의미론적(Semantic) 검색의 한계를 극복하기 위해, 전통 키워드 검색(BM25)과 밀집 벡터 검색(Chroma)을 결합합니다.
  - LangChain의 `EnsembleRetriever`를 활용하여 상호 독점적 순위(RRF, Reciprocal Rank Fusion) 알고리즘을 실습하고, 가중치 조절에 따른 검색 품질 변화를 분석합니다.
- **Senior Mentor의 핵심 가이드**:
  - 키워드 매칭(고유명사, 품번 등)과 의미적 유사성(맥락 이해)이 모두 필요한 실무 비즈니스 질문을 정의하여 하이브리드 검색의 필요성을 실증합니다.
  - 단순 결합을 넘어 각 Retriever의 가중치(weights) 분배가 최종 RAG 답변 품질에 미치는 영향(Trade-off)을 평가합니다.

In [3]:
# 1. 프로젝트 경로 추가 및 환경 설정 로드
import sys
from pathlib import Path

# 현재 작업 디렉토리의 상위(프로젝트 루트)를 Python path에 추가
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import CONTENT_DIR, GOOGLE_AI_API_KEY
print(f"[상태] 프로젝트 루트 경로: {project_root}")
print(f"[상태] Gemini API 키 로드 여부: {'성공' if GOOGLE_AI_API_KEY else '실패'}")

[상태] 프로젝트 루트 경로: /home/hong/project/ai-camp-note
[상태] Gemini API 키 로드 여부: 성공


In [17]:
# 2. 임베딩 모델 및 LLM 구성
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.chat_models import init_chat_model

embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2',
    output_dimensionality=256,
    api_key=GOOGLE_AI_API_KEY
)

CHAT_MODEL = 'google_genai:gemma-4-31b-it'
llm = init_chat_model(CHAT_MODEL, api_key=GOOGLE_AI_API_KEY, temperature=0.1)
print(f"[준비] 256차원 임베딩 및 {CHAT_MODEL} LLM이 준비되었습니다.")

[준비] 256차원 임베딩 및 google_genai:gemma-4-31b-it LLM이 준비되었습니다.


In [28]:
# 3. 실습용 문서 데이터 구성
from langchain_core.documents import Document

# 고유명사(품번, 부서명)와 의미적 뉘앙스가 혼재된 문서 세트 정의
policy_docs = [
    Document(
        page_content="SK-PET-2026 규정에 따르면, 외근 및 출장 시 법인카드 경비 신청서(서식 4호)는 영수증 결제일로부터 3영업일 이내에 제출해야 한다.",
        metadata={"source": "expense_policy.md", "section": "경비", "code": "SK-PET-2026"}
    ),
    Document(
        page_content="정기 보안 교육 수료 기준은 온라인 강의 진도율 100% 달성 및 최종 테스트 80점 이상 획득이며, 미이수자는 사내망 접속이 제한된다.",
        metadata={"source": "security_policy.md", "section": "교육", "code": "SEC-EDU-01"}
    ),
    Document(
        page_content="법인차량 운행 일지 기록 의무: 업무용 차량을 운행한 직원은 반납 전 운행 거리, 목적지, 주유 여부를 전산 시스템에 즉시 입력해야 한다.",
        metadata={"source": "car_policy.md", "section": "차량", "code": "CAR-OP-10"}
    ),
    Document(
        page_content="원격 근무(재택근무) 승인 요건: 주 2회 이하로 신청 가능하며, 전일 17시까지 메신저 및 그룹웨어를 통해 부서장 사전 승인을 득해야 한다.",
        metadata={"source": "hr_policy.md", "section": "인사", "code": "HR-TELE-02"}
    ),
    Document(
        page_content="외부 방문객 출입 통제 지침: 보안 구역(R&D 센터, 전산실) 방문 시에는 최소 1일 전 보안팀 승인을 얻고 안내원 동반 하에만 출입이 가능하다.",
        metadata={"source": "security_policy.md", "section": "보안", "code": "SEC-VISIT-05"}
    )
]
print(f"[준비] 총 {len(policy_docs)}개의 사내 규정 문서가 준비되었습니다.")

[준비] 총 5개의 사내 규정 문서가 준비되었습니다.


In [29]:
# 4. 이종 Retriever 구축 (Chroma vs BM25)
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever

# 4.1. Chroma Vector Store 기반의 밀집 벡터 Retriever (의미론적 검색)
vectorstore = Chroma.from_documents(
    documents=policy_docs,
    embedding=embeddings
)
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 4.2. BM25 기반의 희소 벡터 Retriever (키워드 매칭 검색)
# 한국어 형태소 분석기 없이 기본 띄어쓰기 토크나이저 기준으로 BM25 초기화
keyword_retriever = BM25Retriever.from_documents(policy_docs)
keyword_retriever.k = 3

print("[성공] Semantic Retriever와 Keyword(BM25) Retriever 구축 완료.")

[성공] Semantic Retriever와 Keyword(BM25) Retriever 구축 완료.


In [30]:
# 5. 개별 Retriever 검색 결과 및 한계 비교 분석

# 5.1. 고유명사(품번 코드)가 중요한 질문
query_code = "SK-PET-2026 서식 제출 기한은?"
print(f"\n[질문 1] {query_code}")

print("\n--- [Semantic Retriever 결과] ---")
for doc in semantic_retriever.invoke(query_code):
    print(f"코드: {doc.metadata.get('code')} | 내용: {doc.page_content[:80]}")

print("\n--- [BM25 Retriever 결과] ---")
for doc in keyword_retriever.invoke(query_code):
    print(f"코드: {doc.metadata.get('code')} | 내용: {doc.page_content[:80]}")


[질문 1] SK-PET-2026 서식 제출 기한은?

--- [Semantic Retriever 결과] ---
코드: SK-PET-2026 | 내용: SK-PET-2026 규정에 따르면, 외근 및 출장 시 법인카드 경비 신청서(서식 4호)는 영수증 결제일로부터 3영업일 이내에 제출해야 한다.
코드: HR-TELE-02 | 내용: 원격 근무(재택근무) 승인 요건: 주 2회 이하로 신청 가능하며, 전일 17시까지 메신저 및 그룹웨어를 통해 부서장 사전 승인을 득해야 한다.
코드: SEC-EDU-01 | 내용: 정기 보안 교육 수료 기준은 온라인 강의 진도율 100% 달성 및 최종 테스트 80점 이상 획득이며, 미이수자는 사내망 접속이 제한된다.

--- [BM25 Retriever 결과] ---
코드: SK-PET-2026 | 내용: SK-PET-2026 규정에 따르면, 외근 및 출장 시 법인카드 경비 신청서(서식 4호)는 영수증 결제일로부터 3영업일 이내에 제출해야 한다.
코드: SEC-VISIT-05 | 내용: 외부 방문객 출입 통제 지침: 보안 구역(R&D 센터, 전산실) 방문 시에는 최소 1일 전 보안팀 승인을 얻고 안내원 동반 하에만 출입이 가능하
코드: HR-TELE-02 | 내용: 원격 근무(재택근무) 승인 요건: 주 2회 이하로 신청 가능하며, 전일 17시까지 메신저 및 그룹웨어를 통해 부서장 사전 승인을 득해야 한다.


In [31]:
# 6. EnsembleRetriever를 이용한 Hybrid Search 구현
from langchain_classic.retrievers import EnsembleRetriever

# 상호 독점적 순위(RRF) 알고리즘을 사용한 앙상블 검색
# 가중치를 각각 0.5로 설정하여 균등하게 반영
ensemble_retriever = EnsembleRetriever(
    retrievers=[semantic_retriever, keyword_retriever],
    weights=[0.5, 0.5]
)

print("[성공] Ensemble Retriever 구성 완료 (weights=[0.5, 0.5])")

[성공] Ensemble Retriever 구성 완료 (weights=[0.5, 0.5])


In [32]:
# 7. 하이브리드 검색 성능 실증
query_hybrid = "SK-PET-2026 차량 운행 일지 및 경비 신청 방법"
print(f"\n[질문 2] {query_hybrid}")

print("\n--- [Ensemble Retriever 결과] ---")
for doc in ensemble_retriever.invoke(query_hybrid):
    print(f"코드: {doc.metadata.get('code')} | 내용: {doc.page_content[:80]}")


[질문 2] SK-PET-2026 차량 운행 일지 및 경비 신청 방법

--- [Ensemble Retriever 결과] ---
코드: SK-PET-2026 | 내용: SK-PET-2026 규정에 따르면, 외근 및 출장 시 법인카드 경비 신청서(서식 4호)는 영수증 결제일로부터 3영업일 이내에 제출해야 한다.
코드: CAR-OP-10 | 내용: 법인차량 운행 일지 기록 의무: 업무용 차량을 운행한 직원은 반납 전 운행 거리, 목적지, 주유 여부를 전산 시스템에 즉시 입력해야 한다.
코드: HR-TELE-02 | 내용: 원격 근무(재택근무) 승인 요건: 주 2회 이하로 신청 가능하며, 전일 17시까지 메신저 및 그룹웨어를 통해 부서장 사전 승인을 득해야 한다.


In [33]:
# 8. Hybrid RAG 체인 구축 및 검증
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# RAG 프롬프트 정의
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 사내 규정 도우미 AI다. 아래 제공된 참고 자료만을 활용하여 답변하라. 관련이 없는 내용은 제외하고, 없는 사실은 지어내지 말라."),
    ("user", "[참고 자료]\n{context}\n\n[질문]\n{question}")
])

def format_docs(docs):
    return "\n\n".join(f"- {doc.page_content} (출처: {doc.metadata.get('source')})" for doc in docs)

# RAG 파이프라인 구성
rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 실제 실행 및 답변 확인
question = "SK-PET-2026에 명시된 경비 신청 기한과 보안구역 R&D 센터 출입 요건을 알려줘."
print(f"\n[질문] {question}")
print("\n--- [RAG 답변] ---")
print(rag_chain.invoke(question))


[질문] SK-PET-2026에 명시된 경비 신청 기한과 보안구역 R&D 센터 출입 요건을 알려줘.

--- [RAG 답변] ---
- **경비 신청 기한**: SK-PET-2026 규정에 따라, 외근 및 출장 시 법인카드 경비 신청서(서식 4호)를 영수증 결제일로부터 3영업일 이내에 제출해야 합니다.
- **R&D 센터 출입 요건**: 보안 구역인 R&D 센터 방문 시에는 최소 1일 전 보안팀의 승인을 얻어야 하며, 안내원 동반 하에만 출입이 가능합니다.


In [10]:
# 에이전트 그래프를 콘솔에 ASCII 문자로 시각화
print(rag_chain.get_graph().draw_ascii())


           +---------------------------------+          
           | Parallel<context,question>Input |          
           +---------------------------------+          
                   ***               ***                
                ***                     ***             
              **                           ***          
+-------------------+                         **        
| EnsembleRetriever |                          *        
+-------------------+                          *        
          *                                    *        
          *                                    *        
          *                                    *        
   +-------------+                      +-------------+ 
   | format_docs |                      | Passthrough | 
   +-------------+*                     +-------------+ 
                   ***               ***                
                      ***         ***                   
                         **    

In [1]:
import shutil


print(shutil.which("ffmpeg"))
print(shutil.which("ffprobe"))

/usr/bin/ffmpeg
/usr/bin/ffprobe


In [4]:
from pydub import AudioSegment

# mp3 → wav 변환
sound = AudioSegment.from_mp3(CONTENT_DIR / "나얼 - 방가방가 햄토리 (방가방가 햄토리 OP.) - 에라이스튜디오 (192k).mp3")
sound.export("stt_test.wav", format="wav")

/home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


<_io.BufferedRandom name='stt_test.wav'>

In [5]:
import speech_recognition as sr

recognizer = sr.Recognizer()

# wav 파일 불러오기
with sr.AudioFile("stt_test.wav") as source:
    audio = recognizer.record(source)

# 구글 STT API로 인식
text = recognizer.recognize_google(audio, language="ko-KR")

print(text)

널 살게 잠이 어디든지 달려가요 센조이 좋아하는 사람은 해바라기 방가방가 우리 친구야 빙글빙글 돌아가요 하면서 우리 친구들


In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = CONTENT_DIR / "부영그룹.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(documents)

[Document(metadata={'producer': 'Adobe PDF Library 9.9', 'creator': 'Adobe InDesign CS5 (7.0)', 'creationdate': '2024-02-29T17:43:53+09:00', 'moddate': '2024-02-29T17:43:53+09:00', 'source': '/home/hong/project/ai-camp-note/content/부영그룹.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='17Ins I ght Korea ˚ March 2024\n한 직원자녀 70여명에게 직접적 인 경제지원이 이뤄지도\n록 출산장려금 1억원씩 총 70억원을 지급키로 했다”고 \n밝혔다. \n출산장려금을 지급하는 회사들은 종종 있어왔지만 지원\n금이 억대에 이르는 경우는 부영그룹이 최초다. \n이를 두고 일각에서는 기업 형편에 비해 과도한 복지로 \n재무부담을 키우는 것 아니냐는 우려도 제기된다. 부영\n그룹은 부동산 시장 침체로 인해서 시공능력순위가 1년 \n사이에 35위에서 93위로 급락했다. 주력 사업인 주택부\n문 매출액이 급감한 여파다. 지난 2022년 기준 부영그룹\n의 매출액은 5564억원, 영업손실 1615억원을 기록했다. \n1년 사이에 매출액은 62,7% 줄어들고, 영업손실이 늘어\n난 것이다. 부영그룹 산하 계열사들의 재무건전성에도 비\n상등이 켜졌다. 부영주택과 동광주택산업의 부채비율은 \n각각 437%, 330%에 이른다. 남광건설산업과 남양개발\n도 적자 누적으로 자본잠식에 빠진 상태다.\n문제는 이렇다 할 신수종 사업이 없다는 점이다. 포트폴\n리오 다각화를 위해서 리조트 등 관광부문을 키우려고 \n하지만 신통치 않은 상태다. 올해도 주택시장이 좋지 않\n을 것으로 예상되는 가운데, 실적을 견인할 비전이 부족\n한 상황에서 출산장려금 선심보다 내실다지기가 우선돼\n야 하는 것 아니냐는 지적이다.  \n

In [12]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# BeautifulSoup에게 제목(.media_end_head_title)과 본문(#dic_area) 태그만 파싱하라고 지정
bs_strainer = bs4.SoupStrainer(
    id=["title_area", "dic_area"]
)

website_loader = WebBaseLoader(
    "https://n.news.naver.com/article/437/0000378416",
    bs_kwargs={"parse_only": bs_strainer}
)
website_documents = website_loader.load()
website_documents

[Document(metadata={'source': 'https://n.news.naver.com/article/437/0000378416'}, page_content="출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책\n[앵커]올해 아이 낳을 계획이 있는 가족이라면 솔깃할 소식입니다. 정부가 저출생 대책으로 매달 주는 부모 급여, 0세 아이는 100만원으로 올렸습니다. 여기에 첫만남이용권, 아동수당까지 더하면 아이 돌까지 1년 동안 1520만원을 받습니다. 지자체도 경쟁하듯 지원에 나섰습니다. 인천시는 새로 태어난 아기, 18살될 때까지 1억원을 주겠다. 광주시도 17살될 때까지 7400만원 주겠다고 했습니다. 선거 때면 나타나서 아이 낳으면 현금 주겠다고 밝힌 사람이 있었죠. 과거에는 표만 노린 '황당 공약'이라는 비판이 따라다녔습니다. 그런데 지금은 출산율이 이보다 더 나쁠 수 없다보니, 이런 현금성 지원을 진지하게 정책화 하는 상황까지 온 겁니다. 게다가 기업들도 뛰어들고 있습니다. 이번에는 출산한 직원에게 단번에 1억원을 주겠다는 회사까지 나타났습니다.이상화 기자가 취재했습니다.[기자]한 그룹사가 오늘 파격적인 저출생 정책을 내놨습니다.2021년 이후 태어난 직원 자녀에 1억원씩, 총 70억원을 지원하고 앞으로도 이 정책을 이어가기로 했습니다.해당 기간에 연년생과 쌍둥이 자녀가 있으면 총 2억원을 받게 됩니다.[오현석/부영그룹 직원 : 아이 키우는 데 금전적으로 많이 힘든 세상이잖아요. 교육이나 생활하는 데 큰 도움이 될 거라 생각합니다.]만약 셋째까지 낳는 경우엔 국민주택을 제공하겠다는 뜻도 밝혔습니다.[이중근/부영그룹 회장 : 3년 이내에 세 아이를 갖는 분이 나올 것이고 따라서 주택을 제공할 수 있는 계기가 될 것으로 생각하고.][조용현/부영그룹 직원 : 와이프가 셋째도 갖고 싶어 했는데 경제적 부담 때문에 부정적이었거든요. (이제) 긍정적으로 생각할 수 있을 것 같습니다.]오늘 행사에서는, 회사가 제공하는 출산장려금은 받는 직원들의 세

In [13]:
import re
from langchain_core.documents import Document

# 1. 텍스트 정제 함수 정의
def preprocess_documents(docs: list[Document]) -> list[Document]:
    cleaned_docs = []
    for doc in docs:
        text = doc.page_content
        
        # 줄바꿈 압축
        text = re.sub(r'\n+', '\n', text)
        text = re.sub(r'\s+', ' ', text)  # 모든 공백을 단일 스페이스로 합침 (문맥 연결성 극대화)
        
        # 양끝 공백 제거
        text = text.strip()
        
        # 메타데이터 정리 (불필요하게 긴 크롤링 정보 간소화)
        cleaned_meta = {
            "source": doc.metadata.get("source"),
            "title": doc.metadata.get("title", "PDF Document")
        }
        
        cleaned_docs.append(Document(page_content=text, metadata=cleaned_meta))
    return cleaned_docs

# 2. 합쳐진 문서 정제 실행
all_documents = website_documents + documents
cleaned_documents = preprocess_documents(all_documents)

# 정제 결과 확인
for i, d in enumerate(cleaned_documents):
    print(f"\n[문서 {i+1} 출처]: {d.metadata['source']}")
    print(f"[길이]: {len(d.page_content)}자")
    print(f"[내용 일부]: {d.page_content[:200]}...")



[문서 1 출처]: https://n.news.naver.com/article/437/0000378416
[길이]: 1187자
[내용 일부]: 출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책 [앵커]올해 아이 낳을 계획이 있는 가족이라면 솔깃할 소식입니다. 정부가 저출생 대책으로 매달 주는 부모 급여, 0세 아이는 100만원으로 올렸습니다. 여기에 첫만남이용권, 아동수당까지 더하면 아이 돌까지 1년 동안 1520만원을 받습니다. 지자체도 경쟁하듯 지원에 나섰습니다. 인천시는 새로 태어난 ...

[문서 2 출처]: /home/hong/project/ai-camp-note/content/부영그룹.pdf
[길이]: 1543자
[내용 일부]: 17Ins I ght Korea ˚ March 2024 한 직원자녀 70여명에게 직접적 인 경제지원이 이뤄지도 록 출산장려금 1억원씩 총 70억원을 지급키로 했다”고 밝혔다. 출산장려금을 지급하는 회사들은 종종 있어왔지만 지원 금이 억대에 이르는 경우는 부영그룹이 최초다. 이를 두고 일각에서는 기업 형편에 비해 과도한 복지로 재무부담을 키우는 것 아니냐는 ...


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 한국어 문장은 평균적으로 300~500자 단위가 정보 밀도가 가장 좋습니다.
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", " ", ""]
)

recursive_chunks = recursive_splitter.split_documents(cleaned_documents)
print(f"[Recursive] 생성된 청크 수: {len(recursive_chunks)}개")


[Recursive] 생성된 청크 수: 9개


In [ ]:
# 설치가 완료된 경우 실행
from langchain_experimental.text_splitter import SemanticChunker

# 현재 노트북 위에 선언되어 있는 embeddings 객체 주입
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"  # 유사도의 상위 백분율 기준으로 경계 분할
)

semantic_chunks = semantic_splitter.split_documents(cleaned_documents)
print(f"[Semantic] 생성된 청크 수: {len(semantic_chunks)}개")

# 의미론적 청킹 결과의 문맥 보존 확인
for i, chunk in enumerate(semantic_chunks):
    print(f"\n--- [의미론적 청크 {i+1}] (길이: {len(chunk.page_content)}자) ---")
    print(chunk.page_content)


[Semantic] 생성된 청크 수: 5개

--- [의미론적 청크 1] (길이: 392자) ---
출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책 [앵커]올해 아이 낳을 계획이 있는 가족이라면 솔깃할 소식입니다. 정부가 저출생 대책으로 매달 주는 부모 급여, 0세 아이는 100만원으로 올렸습니다. 여기에 첫만남이용권, 아동수당까지 더하면 아이 돌까지 1년 동안 1520만원을 받습니다. 지자체도 경쟁하듯 지원에 나섰습니다. 인천시는 새로 태어난 아기, 18살될 때까지 1억원을 주겠다. 광주시도 17살될 때까지 7400만원 주겠다고 했습니다. 선거 때면 나타나서 아이 낳으면 현금 주겠다고 밝힌 사람이 있었죠. 과거에는 표만 노린 '황당 공약'이라는 비판이 따라다녔습니다. 그런데 지금은 출산율이 이보다 더 나쁠 수 없다보니, 이런 현금성 지원을 진지하게 정책화 하는 상황까지 온 겁니다.

--- [의미론적 청크 2] (길이: 794자) ---
게다가 기업들도 뛰어들고 있습니다. 이번에는 출산한 직원에게 단번에 1억원을 주겠다는 회사까지 나타났습니다.이상화 기자가 취재했습니다.[기자]한 그룹사가 오늘 파격적인 저출생 정책을 내놨습니다.2021년 이후 태어난 직원 자녀에 1억원씩, 총 70억원을 지원하고 앞으로도 이 정책을 이어가기로 했습니다.해당 기간에 연년생과 쌍둥이 자녀가 있으면 총 2억원을 받게 됩니다.[오현석/부영그룹 직원 : 아이 키우는 데 금전적으로 많이 힘든 세상이잖아요. 교육이나 생활하는 데 큰 도움이 될 거라 생각합니다.]만약 셋째까지 낳는 경우엔 국민주택을 제공하겠다는 뜻도 밝혔습니다.[이중근/부영그룹 회장 : 3년 이내에 세 아이를 갖는 분이 나올 것이고 따라서 주택을 제공할 수 있는 계기가 될 것으로 생각하고.][조용현/부영그룹 직원 : 와이프가 셋째도 갖고 싶어 했는데 경제적 부담 때문에 부정적이었거든요. (이제) 긍정적으로 생각할 수 있을 것 같습니다.]오늘 행사에서는, 회사가 제공하는 출산장려금은 받는 직원들의 세금 부담을 고려해

In [22]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 자식 스플리터만 정의
child_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=15)

vectorstore = Chroma(collection_name="split_parents_b", embedding_function=embeddings)
docstore = InMemoryStore()

# 2. parent_splitter 를 None 으로 설정
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=None, # None으로 비워둡니다.
)

# 3. 우리가 앞서 의미론적으로 쪼개둔 'semantic_chunks'를 직접 주입
# retriever가 이 청크들을 '부모 문서'로 인식하고, 내부에서 '자식 스플리터'를 돌려 매핑해 줍니다.
retriever.add_documents(semantic_chunks)


In [ ]:
# 1. 테스트 질문 정의
query = "출산장려금 1억원 지급 시 소득세와 증여세 세금 논쟁에 대해 알려줘."

# 2. 검색 수행
retrieved_docs = retriever.invoke(query)

print(f"[검색된 문서 수]: {len(retrieved_docs)}개")
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- [호출된 부모 문서 {i+1}] (길이: {len(doc.page_content)}자) ---")
    print(doc.page_content)


[검색된 문서 수]: 2개

--- [호출된 부모 문서 1] (길이: 654자) ---
포트폴 리오 다각화를 위해서 리조트 등 관광부문을 키우려고 하지만 신통치 않은 상태다. 올해도 주택시장이 좋지 않 을 것으로 예상되는 가운데, 실적을 견인할 비전이 부족 한 상황에서 출산장려금 선심보다 내실다지기가 우선돼 야 하는 것 아니냐는 지적이다. 이번 부영의 출산장려금에는 조세 문제가 뜨거운 감자로 떠오르면서 이와 관련한 논의도 진행되고 있다. 이에 대 한 논의가 잘 이뤄질 경우 부영그룹의 선례가 남아 향후 다른 기업들의 출산장려금에도 적용될 수 있을 것으로 전망된다. 부영그룹이 쏘아올린 ‘세제 혜택’ 이슈 부영그룹은 출산장려금 1억원을 지원할 때 ‘직원’이 아닌 ‘자녀’에게 증여하는 방법을 선택했다. 이로 인해 출산장 려금을 근로소득으로 봐야하느냐, 증여로 봐야하느냐 하는 논쟁이 불거졌다. 현재 근로소득 과세표준 구간을 보면 ▲1400만원 이하 6% ▲1400만원 초과~5000만원 이하 15% ▲5000만원 초과 8800만원 이하 24% ▲8800만원 초과 1억5000만 원 이하 35% ▲1억5000만원 초과 3억 이하 38% 등으로 구간이 나눠져 있다. 출산장려금 1억원이 근로소득으로 들어갈 경우 직원들의 연봉과 더해져 세후 실수령 액은 대폭 줄어들 것으로 예상된다. 반면에 증여로 인정될 경우 1억원 이하 증여에 대한 세율 10%만 적용된다.

--- [호출된 부모 문서 2] (길이: 369자) ---
직원 입장에서 세부담이 크게 줄어드 는 셈이다. 이에 이 회장은 지난 2월 19일 이기일 보건복지부 제1차 관을 만나 출산장려금을 지원하게 된 계기를 설명하고 이에 대한 면세 혜택 도입을 당부했다. 정부 역시도 출산장려금이 저출산 대책인 것을 고려해 긍정적인 방향으로 검토하고 있는 것으로 알려졌다. 2월 5일 서울시 중구에 위치한 부영태평빌딩 컨벤션홀에서 열린 2024년 시무식 후 출산장려금을 받은 임직원들이 기념촬영을 하고 있다.<인사이트코리아> 부영그룹은 202

In [35]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# 1. 출처 보존형 RAG 체인 설계
# 쿼리가 들어오면 문서를 먼저 검색하여 보관하고, 그 문서를 가공하여 LLM에 전달합니다.
rag_chain_with_sources = (
    RunnableParallel(
        {"documents": retriever, "question": RunnablePassthrough()}
    )
    .assign(context=lambda x: format_docs(x["documents"]))
    .assign(answer=rag_prompt | llm | StrOutputParser())
)

# 2. 실행 및 결과 검증
question = "부영그룹이 출산장려금 1억원을 지급할 때 발생한 세금 문제와 이에 대한 정부의 검토 방향을 요약해줘."
result = rag_chain_with_sources.invoke(question)

# 3. 정돈된 출력
print(f"[질문]: {question}\n")
print("--- [RAG 최종 답변] ---")
print(result["answer"])

print("\n--- [답변의 출처 목록] ---")
# 중복 제거된 부모 문서들의 메타데이터에서 source 추출
for i, doc in enumerate(result["documents"]):
    source = doc.metadata.get("source", "출처 없음")
    title = doc.metadata.get("title", "알 수 없는 문서")
    print(f"[{i+1}] {title} (URL/경로: {source})")


[질문]: 부영그룹이 출산장려금 1억원을 지급할 때 발생한 세금 문제와 이에 대한 정부의 검토 방향을 요약해줘.

--- [RAG 최종 답변] ---
부영그룹의 출산장려금 지급과 관련한 세금 문제와 정부의 검토 방향은 다음과 같습니다.

*   **세금 문제:** 부영그룹이 출산장려금 1억원을 '직원'이 아닌 '자녀'에게 증여하는 방식을 선택함에 따라, 이를 **근로소득**으로 볼 것인지 아니면 **증여**로 볼 것인지에 대한 논쟁이 발생했습니다. 근로소득으로 간주될 경우 연봉과 합산되어 높은 과세표준 구간이 적용되어 실수령액이 크게 줄어들지만, 증여로 인정될 경우 1억원 이하 증여 세율인 10%만 적용되어 직원의 세부담이 줄어들게 됩니다.
*   **정부의 검토 방향:** 정부는 출산장려금이 저출산 대책이라는 점을 고려하여, 면세 혜택 도입 요청에 대해 긍정적인 방향으로 검토하고 있는 것으로 알려졌습니다.

--- [답변의 출처 목록] ---
[1] PDF Document (URL/경로: /home/hong/project/ai-camp-note/content/부영그룹.pdf)
[2] PDF Document (URL/경로: /home/hong/project/ai-camp-note/content/부영그룹.pdf)


In [37]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# 1. 평가 결과를 규격화하여 받아낼 Pydantic 스키마 정의
class RAGEvaluation(BaseModel):
    score: int = Field(description="RAG 답변의 품질 점수 (1점부터 5점까지)")
    reasoning: str = Field(description="이 점수를 부여한 구체적인 사실적 근거 및 개선 제안")

# 2. 커스텀 평가 감사관 프롬프트 정의
eval_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "너는 사내 RAG 시스템의 답변 품질을 검증하는 전문 감사관이다. "
     "제공된 [참고 자료]만을 바탕으로 [답변]이 [질문]에 대해 사실에 입각하여 정확하고 친절하게 답변했는지 엄격히 평가하라.\n\n"
     "이하의 5단계 척도를 기준으로 삼아라:\n"
     "- 5점: 답변이 사실에 완벽히 부합하고 질문의 요점을 모두 친절히 설명함.\n"
     "- 3점: 거짓 정보는 없으나 참고 자료의 내용이 누락되었거나 질문을 비껴감.\n"
     "- 1점: 참고 자료에 없는 내용을 상상해서 답변함 (할루시네이션) 혹은 심각한 동문서답.\n\n"
     "평가 결과를 지정된 구조(JSON)로 정확히 출력하라."
    ),
    ("user", "[참고 자료]\n{context}\n\n[질문]\n{question}\n\n[답변]\n{answer}")
])

# 3. 랭체인 모델에 구조화된 아웃풋 주입하여 평가 체인 완성
custom_evaluator = eval_prompt | llm.with_structured_output(RAGEvaluation)

# 4. 앞서 실행했던 RAG 결과(result)를 대입하여 채점 실행
eval_result = custom_evaluator.invoke({
    "context": format_docs(result["documents"]),
    "question": question,
    "answer": result["answer"]
})

# 5. 정돈된 결과 확인
print(f"■ RAG 답변 최종 점수: {eval_result.score} / 5")
print(f"■ 감사관의 평가 근거:\n{eval_result.reasoning}")


■ RAG 답변 최종 점수: 5 / 5
■ 감사관의 평가 근거:
답변은 제공된 참고 자료의 핵심 내용을 정확하게 요약하였습니다. 세금 문제의 쟁점(근로소득 vs 증여)과 그에 따른 세액 차이의 근거, 그리고 정부의 긍정적인 검토 방향까지 모두 사실에 입각하여 친절하게 설명하였습니다.
